# Direct-beam distance calibration

Use the headless `xrd_tools` API with explicit angles/images, then with a SPEC scan.
Run in a kernel containing the direct-beam implementation (`xdart[notebook]` supplies
`ipympl`). Before merging a feature checkout, start Jupyter with that checkout's
`src` on `PYTHONPATH`; this notebook does not change your Python installation.

The tangent model fits only intercept and signed scale, with a **known** motor zero
where the detector normal is parallel to the beam. Distance is perpendicular
sample-to-plane separation for a flat detector on a sample-centred arm.
The beam pixel is an approximate strip locator, never a fixed geometric constraint.
Pixels are zero-based `(x,y)=(column,row)`; arrays are `(row,column)`.


In [ ]:
%matplotlib widget

from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from xrd_tools.integrate import calibrate_direct_beam, calibrate_direct_beam_scan


## Explicit arrays: unequal, reversed angles and a missing frame

This small synthetic detector has rectangular pixels. The example deliberately
keeps an interior missing image as `None`, preserving its motor angle and identifier.
Replace these arrays with your own ordered images, paths, or `(path, frame, dataset)`
references. Horizontal motion uses column pitch; vertical uses row pitch.


In [ ]:
angles_deg = np.array([8.1, 7.7, 7.13, 7.0, 6.82, 6.2, 5.7])
known_zero_deg = 7.0
known_distance_m = 0.080
row_pitch_m, column_pitch_m = 120e-6, 80e-6
shape = (45, 161)
beam_pixel = (80, 22)
yy, xx = np.indices(shape)
centres = beam_pixel[0] - known_distance_m / column_pitch_m * np.tan(
    np.deg2rad(angles_deg - known_zero_deg))
images = [12 + 4000*np.exp(-0.5*(((xx-centre)/1.6)**2 + ((yy-beam_pixel[1])/1.6)**2))
          for centre in centres]
images[2] = None

synthetic = calibrate_direct_beam(
    angles_deg, iter(images),
    detector="Detector",
    detector_config={"pixel1": row_pitch_m, "pixel2": column_pitch_m, "max_shape": shape},
    beam_pixel=beam_pixel, motion="horizontal", strip_half_width=2,
    angle_zero_deg=known_zero_deg,
    frame_ids=[f"exposure-{i:03d}" for i in range(len(angles_deg))],
)
print(f"Distance: {synthetic.distance_mm:.6f} mm (truth: {known_distance_m*1000:.3f} mm)")
print(f"Regression standard error: {synthetic.distance_std_m*1000:.6g} mm")
print("Rejected:", [(synthetic.frame_ids[i], why)
                    for i, why in enumerate(synthetic.rejection_reasons) if why])


## SPEC scan and raw images

Set the data root below. For the local example folder, `../test_data` names the
existing dataset. The scan identifier is `number.repetition`, not a list index.
These files are Pilatus100k, `(195,487)`, little-endian int32 with no header.
The `del` column spans −1 to +1 degrees. Here zero is **assumed** to be the normal
incidence position; verify that convention for your experiment.

Optional reader settings also support HDF5 `dataset_path`/`frame`; explicit image
references can instead be passed as `images=...`. No filename discovery or angle
reconstruction is performed. A missing file becomes a rejected original row.


In [ ]:
DATA_ROOT = Path(os.environ.get("XDART_TEST_DATA", "../test_data")).expanduser()
scan_file = DATA_ROOT / "RSM" / "direct_beam"
real = None
if scan_file.is_file():
    real = calibrate_direct_beam_scan(
        scan_file, "1.1", motor="del",
        image_dir=DATA_ROOT / "RSM" / "images",
        filename_template="b_thampy_direct_beam_scan{scan}_{point:04d}.raw",
        point_index_origin=0,
        detector="Pilatus100k", beam_pixel=(243, 99), motion="horizontal",
        angle_zero_deg=0.0,
        reader_options={"detector_shape": (195, 487), "raw_dtype": "<i4",
                        "raw_header_skip": 0},
        # saturation=...  # Supply the acquisition's inclusive clipping limit if known.
    )
    print(f"Distance: {real.distance_mm:.3f} mm")
    print(f"Regression standard error: {real.distance_std_m*1000:.3f} mm")
    print(f"Used: {real.used.sum()}/{len(real.used)}, motion sign: {real.motion_sign:+d}")
    print("Rejected:", [(real.point_ids[i], why)
                        for i, why in enumerate(real.rejection_reasons) if why])
else:
    print("Set DATA_ROOT to run the real SPEC example; synthetic diagnostics remain available.")


## Inspect centres and residuals

Points retain original angle/frame pairing. Residuals are measured minus predicted
centres. Fit uncertainty excludes unknown pitch, angle-zero, tilt and saturation
errors. In particular, data below int32 maximum need not be physically unsaturated.
No PONI, wavelength or full detector orientation has been calibrated.


In [ ]:
result = real if real is not None else synthetic
used = result.used
order = np.argsort(result.angles_deg)
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7, 6), layout="constrained")
axes[0].plot(result.angles_deg[used], result.centres_px[used], "o", label="Gaussian centres")
axes[0].plot(result.angles_deg[order], result.predictions_px[order], "-", label=result.model)
if (~used).any():
    axes[0].plot(result.angles_deg[~used], result.predictions_px[~used], "x",
                 label="Rejected row (prediction only)")
axes[0].set_ylabel("Beam centre (pixel)")
axes[0].legend()
axes[1].plot(result.angles_deg[used], result.residuals_px[used], "o")
axes[1].axhline(0, color="gray", lw=1)
axes[1].set(xlabel="Detector motor angle (degree)", ylabel="Residual (pixel)")
plt.show()


## Compare the notebook's local linear approximation

The legacy method regresses angles against centres and reports
`pitch / tan(abs(slope) * pi/180)`. It agrees with tangent geometry for a narrow
scan near normal incidence. It does not become an exact wide-angle model by
using more points. The tangent fit requires a known zero; it does not refine it.


In [ ]:
linear = calibrate_direct_beam(
    angles_deg, iter(images), detector="Detector",
    detector_config={"pixel1": row_pitch_m, "pixel2": column_pitch_m, "max_shape": shape},
    beam_pixel=beam_pixel, model="linear",
)
print(f"Synthetic tangent: {synthetic.distance_mm:.6f} mm")
print(f"Synthetic notebook approximation: {linear.distance_mm:.6f} mm")
print(f"Signed local angle/position slope: {synthetic.slope_deg_per_pixel:.7f} degree/pixel")
